In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from model.hopfield_network import HopfieldNetwork
from notebooks.plot_patterns import plot_patterns

In [ ]:
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

if project_root not in sys.path:
    sys.path.append(project_root)

In [ ]:
patterns_path = Path(project_root) / "data/OCRA-12x30-cut.csv"

In [ ]:
df = pd.read_csv(patterns_path, header=None)
df.head(20)

In [ ]:
patterns = np.asarray(df, dtype=float)
plot_patterns(patterns, (30, 12))

In [ ]:
n = 360
method = "oji" # hebb oji
if_asynchronic = True # True False
lrs = np.arange(0.0005, 0.0065, 0.0005).tolist()

In [ ]:
if method == "oji":
    best_lr = 0.006
    max_stable = 0
    for lr in lrs:
        hn = HopfieldNetwork(n=n, method=method, if_asynchronic=if_asynchronic, lr=lr, epochs=30, random_state=42)
        hn.train(patterns=patterns)
        stable_patterns = 0
        for i in range(patterns.shape[0]):
            if hn.check_if_stable(patterns[i]):
                stable_patterns += 1
        display(f"Stable patterns for {method} method with learning_rate {lr:.4f}: {stable_patterns}/{patterns.shape[0]}")
        if stable_patterns > max_stable:
            best_lr = lr
            max_stable = stable_patterns

    display(f"Best results for learning_rate {best_lr:.4f}: {max_stable}/{patterns.shape[0]}")
    hn = HopfieldNetwork(n=n, method=method, if_asynchronic=if_asynchronic, lr=best_lr, epochs=30, random_state=42)
    hn.train(patterns=patterns)
else:
    hn = HopfieldNetwork(n=n, method=method, if_asynchronic=if_asynchronic)
    hn.train(patterns=patterns)
    stable_patterns = 0
    for i in range(patterns.shape[0]):
        if hn.check_if_stable(patterns[i]):
            stable_patterns += 1

    display(f"Stable patterns for {method} method: {stable_patterns}/{patterns.shape[0]}")


In [ ]:
hn.train(patterns=patterns)

In [ ]:
for i in range(patterns.shape[0]):
    result = hn.recall(patterns[i], 100)
    display(f"Result: {result["Result"]}. Energy value: {result["Energy"]}")
    hn.plot_history(result["History"], (30, 12))

In [ ]:
noise_levels = np.arange(0, 0.31, 0.02)

rng = np.random.default_rng(123)
hn.rng = np.random.default_rng(42)

results = []

for i in range(patterns.shape[0]):
    original_pattern = patterns[i]

    for noise_level in noise_levels:
        correct_recalled = 0
        stable_recalled = 0
        total_iterations = 0

        for test in range(100):
            noisy_pattern = original_pattern.copy()
            n_flips = round(noise_level * noisy_pattern.size)

            indices = rng.choice(noisy_pattern.size, size=n_flips, replace=False)

            noisy_pattern[indices] *= -1

            result = hn.recall(noisy_pattern, max_iter=100)

            output = result["Output states"]

            if np.array_equal(original_pattern, output):
                correct_recalled += 1

            if result["Result"] == "stable":
                stable_recalled += 1

            total_iterations += result["Last iteration"]

        total_tests = 100

        results.append({
            "Pattern": i,
            "Noise level (%)": noise_level * 100,
            "Correct recall (%)": (
                correct_recalled / total_tests * 100
            ),
            "Stable states (%)": (
                stable_recalled / total_tests * 100
            ),
            "Mean iterations": total_iterations / total_tests,
        })

df_results = pd.DataFrame(results)
display(df_results)


In [ ]:
df_summary = df_results.groupby(
    "Noise level (%)", as_index=False
)[["Correct recall (%)", "Stable states (%)"]].mean()

plt.figure(figsize=(9, 3))

plt.plot(
    df_summary["Noise level (%)"],
    df_summary["Correct recall (%)"],
    marker="o",
    label="Correct recall"
)

plt.plot(
    df_summary["Noise level (%)"],
    df_summary["Stable states (%)"],
    marker="s",
    label="Stable states"
)

plt.xlabel("Noise level (%)")
plt.ylabel("Percentage (%)")
plt.title("Recall accuracy and stable states")
plt.ylim(-5, 105)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))

for pattern_id, group in df_results.groupby("Pattern"):
    plt.plot(
        group["Noise level (%)"],
        group["Correct recall (%)"],
        marker="o",
        label=f"Pattern {pattern_id}"
    )

plt.xlabel("Noise level (%)")
plt.ylabel("Correct recall (%)")
plt.title("Recall accuracy for every pattern")
plt.ylim(-5, 105)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()